# IoT Botnet Detection - Exploratory Data Analysis
## Group 20: Rajat Jaswal, Mangesh Bhattacharya, Riya Kriplani

This notebook performs exploratory data analysis on the N-BaIoT dataset for IoT botnet detection.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add project src to path
sys.path.insert(0, '../src')

# Import custom modules
from data_processing.data_loader import DataLoader
from visualization.visualizer import Visualizer

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 1. Load Dataset

In [ ]:
# Initialize data loader
data_loader = DataLoader('../data/raw')

# Load benign traffic
benign_df = data_loader.load_csv('../data/raw/benign_traffic.csv', label='benign')

print(f"Dataset shape: {benign_df.shape}")
print(f"\nFirst few rows:")
benign_df.head()

## 2. Dataset Information

In [ ]:
# Get feature information
feature_info = data_loader.get_feature_info()
print("Feature Categories:")
for category, count in feature_info.items():
    print(f"  {category}: {count}")

In [ ]:
# Basic statistics
print("Dataset Info:")
print(benign_df.info())

print("\nBasic Statistics:")
benign_df.describe()

## 3. Missing Values Analysis

In [ ]:
# Check for missing values
missing = benign_df.isnull().sum()
if missing.sum() > 0:
    print("Missing values found:")
    print(missing[missing > 0])
else:
    print("✓ No missing values found!")

## 4. Feature Distribution Analysis

In [ ]:
# Plot distribution of first 10 features
features = benign_df.columns[:10]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.ravel()

for idx, feature in enumerate(features):
    axes[idx].hist(benign_df[feature], bins=50, alpha=0.7, color='steelblue')
    axes[idx].set_title(feature, fontsize=10)
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../results/figures/feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Correlation Analysis

In [ ]:
# Calculate correlation matrix for a subset of features
subset_features = benign_df.columns[:30]
correlation_matrix = benign_df[subset_features].corr()

# Plot heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(
    correlation_matrix,
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title('Feature Correlation Heatmap (First 30 Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Summary Statistics by Feature Category

In [ ]:
# Group features by category
categories = {
    'MI': [col for col in benign_df.columns if col.startswith('MI_')],
    'H': [col for col in benign_df.columns if col.startswith('H_') and not col.startswith('HH_')],
    'HH': [col for col in benign_df.columns if col.startswith('HH_') and not col.startswith('HH_jit')],
    'HH_jit': [col for col in benign_df.columns if col.startswith('HH_jit')],
    'HpHp': [col for col in benign_df.columns if col.startswith('HpHp_')]
}

# Summary statistics for each category
for category, features in categories.items():
    if features:
        print(f"\n{category} Features Summary:")
        print(f"Number of features: {len(features)}")
        print(f"Mean of means: {benign_df[features].mean().mean():.4f}")
        print(f"Mean of std: {benign_df[features].std().mean():.4f}")

## 7. Feature Value Ranges

In [ ]:
# Analyze value ranges
value_ranges = pd.DataFrame({
    'min': benign_df.min(),
    'max': benign_df.max(),
    'range': benign_df.max() - benign_df.min(),
    'mean': benign_df.mean(),
    'std': benign_df.std()
})

print("Feature Value Ranges (first 10 features):")
value_ranges.head(10)

## 8. Data Quality Report

In [ ]:
# Validate data
is_valid, issues = data_loader.validate_data(benign_df)

print("Data Quality Report:")
print(f"Data is valid: {is_valid}")
if issues:
    print("\nIssues found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("✓ No issues found!")

## 9. Key Insights

### Dataset Characteristics:
1. **Feature Count**: 115 statistical features derived from network traffic
2. **Feature Categories**: MI (MAC-IP), H (Host), HH (Channel), HH_jit (Jitter), HpHp (Socket)
3. **Time Windows**: L5, L3, L1, L0.1, L0.01 (different decay factors)
4. **Statistics**: Weight, mean, variance, std, magnitude, radius, covariance, pcc

### Next Steps:
1. Load attack data for comparison
2. Perform feature engineering and selection
3. Train machine learning models
4. Evaluate and compare model performance